# 3. Cleaning per participant

Follows `ana03_preprocessing.m` of the reference pipeline, without its epoching (this thesis analyzes whole trials of continuous data, see the end of the notebook). **Run once per participant, after `02_corrmap.ipynb`.**

**Flow:** 1. load the prepared data, the ICA and the components to remove → 2. remove the components from the 500 Hz data → 3. filter: 0.1–40 Hz (MNE's `filter()`) → 4. save the clean data, a report and the summary.

In [ ]:
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config as cfg
%matplotlib inline

## 0. Participant

In [ ]:
import os

# execute_notebooks.py sets RT_SUBJECT
SUBJECT = os.environ.get("RT_SUBJECT", "sub-001")
OUT_DIR = cfg.out_dir(SUBJECT)
print(SUBJECT, f"({cfg.session_of(SUBJECT)})", "->", OUT_DIR)

## 1. Loading

In [ ]:
raw = mne.io.read_raw(OUT_DIR / f"{SUBJECT}_prep_raw.fif", preload=True)
ica = mne.preprocessing.read_ica(OUT_DIR / f"{SUBJECT}_ica.fif")
badcomps = pd.read_csv(OUT_DIR / f"{SUBJECT}_badcomps.csv")
ica.exclude = sorted(int(c) for c in badcomps["component"])
print(f"{SUBJECT}: {raw.info['sfreq']} Hz, {raw.times[-1]:.1f} s; removing components {ica.exclude}")
badcomps

## 2. Removing the artifact components

The ICA was learned on the filtered 250 Hz copy; its weights are applied to the prepared data at 500 Hz (as `ana02` removes the components from the original data with `pop_subcomp`).

In [ ]:
raw_before = raw.copy()
ica.apply(raw)
ica.plot_overlay(raw_before, picks="eeg")

## 3. Filtering

Band-pass 0.1–40 Hz, the cut-offs of `ana03_preprocessing.m`, with MNE's `filter()` instead of the reference's fixed-order EEGLAB filters: the reference's 0.1 Hz high-pass (order 500, 1 s long) hardly filters at all (−0.9 dB at every frequency below 0.5 Hz), so the mV drift of the recordings would stay in the data (see `01_ica`, step 6, and `ica_reference_questions.md` #9). No re-referencing: the data stay in the recording reference, as in the reference pipeline.

In [ ]:
raw.filter(cfg.HP, cfg.LP, picks=["eeg", "eog"])  # 0.1-40 Hz, MNE's default FIR filter

# Browsing the whole recording takes minutes, so it is off by default
# (set SHOW_RAW = True when running this notebook by hand and you want to look)
SHOW_RAW = False
if SHOW_RAW:
    raw.plot(n_channels=30, scalings=dict(eeg=50e-6))

## 4. Saving the clean data, a report and the summary

`sub-<participant>_clean_raw.fif` is the input for the analysis. The HTML report shows the data before and after cleaning (with spectra: the alpha peak around 10 Hz should survive) and the ICA. `preprocessing_summary.csv` collects the numbers for the Methods, one row per participant.

In [ ]:
raw.save(OUT_DIR / f"{SUBJECT}_clean_raw.fif", overwrite=True)

report = mne.Report(title=f"Preprocessing {SUBJECT}")
report.add_raw(raw_before, title="Before cleaning (prepared data)", psd=True)
report.add_ica(ica, title="ICA", inst=raw_before)
report.add_raw(raw, title="After cleaning", psd=True)
report.save(OUT_DIR / f"{SUBJECT}_report.html", overwrite=True, open_browser=False)

info = pd.read_csv(OUT_DIR / f"{SUBJECT}_ica_info.csv", dtype={"subject": str, "bad_channels": str})
info["removed_components"] = " ".join(map(str, ica.exclude))
info["n_removed_components"] = len(ica.exclude)
info["duration_s"] = round(raw.times[-1], 1)

summary_file = cfg.OUT_ROOT / "preprocessing_summary.csv"
if summary_file.exists():
    previous = pd.read_csv(summary_file, dtype={"subject": str})
    info = pd.concat([previous[previous["subject"] != SUBJECT], info])
info.to_csv(summary_file, index=False)
info

## Next: analysis (separate notebook)

The epoching of `ana03` (−0.2 to 0.8 s around a stimulus, joint-probability rejection at 4 SD) is not used: cRQA needs long continuous segments. The analysis works on the clean continuous data:

1. Alpha (8–12 Hz) and beta (13–30 Hz) amplitude envelopes on the continuous data (cutting first would add filter edge artifacts to every segment).
2. Cut pairs of participants into trials **through session time** (`rt_bids.from_session_time` with the start and end from `session_events`), never relative to the file start or to one event per file: the headsets' clocks drift against the session clock and some skipped samples. The clean file was cropped, so subtract its `first_samp` from the sample positions (`rt_bids.crop_session` assumes an uncropped recording).
3. Teammate/opponent label per trial from `team_id` in `session_events` (teams change between blocks).
4. Downsample the envelopes to ~20–50 Hz, then cRQA.